In [1]:
import pandas as pd
import requests
import numpy as np
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
#url de la API
api_url='https://api.yelp.com/v3/businesses/search'

#estos datos corresponden a una cuenta de usuario creada previamente
clientid='GWOCZh9-BmZxtdsAjr7Gug'
apikey='FHVvXoNmTXIl9DuxYis7AV5uLPujm9MLwrhgs5NgvCfaOxd3V6mxt6dQU8eEqYJiGxe816XATx7ufWjbMWqbV-2Uku1jxBJv8BGRC74NroLPl27PDQqs0tDixit-YHYx'
headers={'Authorization':'Bearer %s'%apikey}

In [4]:
ciudad = "Boston" # Se puede elegir cualquier ciudad del dataset entregado

In [5]:
# De esta manera se consulta el API de Yelp para obtener información sobre restaurantes en la ciudad especificada
params={'term':'restaurants','location': ciudad,'limit':50}
response=requests.get(api_url,params=params,headers=headers)
data=response.json()
data

{'businesses': [{'id': 'kP1b-7BO_VhWk_0tvuA_tw',
   'alias': 'carmelinas-boston-2',
   'name': "Carmelina's",
   'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/pvH7ItsbeuRdNFpaXXG_lw/o.jpg',
   'is_closed': False,
   'url': 'https://www.yelp.com/biz/carmelinas-boston-2?adjust_creative=GWOCZh9-BmZxtdsAjr7Gug&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=GWOCZh9-BmZxtdsAjr7Gug',
   'review_count': 4336,
   'categories': [{'alias': 'italian', 'title': 'Italian'}],
   'rating': 4.5,
   'coordinates': {'latitude': 42.363949021285016,
    'longitude': -71.05419142050383},
   'transactions': ['delivery', 'restaurant_reservation'],
   'price': '$$$',
   'location': {'address1': '307 Hanover St',
    'address2': '',
    'address3': '',
    'city': 'Boston',
    'zip_code': '02113',
    'country': 'US',
    'state': 'MA',
    'display_address': ['307 Hanover St', 'Boston, MA 02113']},
   'phone': '+16177420020',
   'display_phone': '(617) 742-0020',
   'distance': 

In [6]:
#Veamos los keys del diccionario recibido
data.keys()

dict_keys(['businesses', 'total', 'region'])

In [7]:
#El primer elemento del diccionario indica el total de restaurants existentes en la API
print('En total la base de datos registra %d restaurants'%data['total'])

En total la base de datos registra 6800 restaurants


In [8]:
data['businesses'] # Hasta acá se le entregaría a los estudiantes

[{'id': 'kP1b-7BO_VhWk_0tvuA_tw',
  'alias': 'carmelinas-boston-2',
  'name': "Carmelina's",
  'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/pvH7ItsbeuRdNFpaXXG_lw/o.jpg',
  'is_closed': False,
  'url': 'https://www.yelp.com/biz/carmelinas-boston-2?adjust_creative=GWOCZh9-BmZxtdsAjr7Gug&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=GWOCZh9-BmZxtdsAjr7Gug',
  'review_count': 4336,
  'categories': [{'alias': 'italian', 'title': 'Italian'}],
  'rating': 4.5,
  'coordinates': {'latitude': 42.363949021285016,
   'longitude': -71.05419142050383},
  'transactions': ['delivery', 'restaurant_reservation'],
  'price': '$$$',
  'location': {'address1': '307 Hanover St',
   'address2': '',
   'address3': '',
   'city': 'Boston',
   'zip_code': '02113',
   'country': 'US',
   'state': 'MA',
   'display_address': ['307 Hanover St', 'Boston, MA 02113']},
  'phone': '+16177420020',
  'display_phone': '(617) 742-0020',
  'distance': 2274.0040366572953},
 {'id': 'aGYdF_fN

In [11]:
# Lista para acumular resultados
resultados = []

# Bucle para traer hasta 200 restaurantes (en bloques de 50)
for offset in range(0, 200, 50):  # 0, 50, 100, 150
    
    params = {
        'term': 'restaurants',
        'location': ciudad,
        'limit': 50,
        'offset': offset
    }
    
    response = requests.get(api_url, params=params, headers=headers)
    
    # Validación básica
    if response.status_code != 200:
        print(f"Error en la request: {response.status_code}")
        break
    
    data = response.json()
    negocios = data.get('businesses', [])
    
    # Si ya no hay más resultados, corta
    if not negocios:
        break
    
    resultados.extend(negocios)

# Convertir a DataFrame
df_restaurantes = pd.json_normalize(resultados)

# Eliminar duplicados por seguridad
df_restaurantes = df_restaurantes.drop_duplicates(subset='id')

# Ver resultados
print(f"Total restaurantes obtenidos: {len(df_restaurantes)}")
df_restaurantes.head()

Total restaurantes obtenidos: 200


,id,alias,name,image_url,is_closed,url,review_count,categories,rating,transactions,price,phone,display_phone,distance,coordinates.latitude,coordinates.longitude,location.address1,location.address2,location.address3,location.city,location.zip_code,location.country,location.state,location.display_address
0,kP1b-7BO_VhWk_0tvuA_tw,carmelinas-boston-2,Carmelina's,https://s3-media0.fl.yelpcdn.com/bphoto/pvH7It...,False,https://www.yelp.com/biz/carmelinas-boston-2?a...,4336,"[{'alias': 'italian', 'title': 'Italian'}]",4.5,"[delivery, restaurant_reservation]",$$$,+16177420020,(617) 742-0020,2274.004037,42.363949,-71.054191,307 Hanover St,,,Boston,02113,US,MA,"[307 Hanover St, Boston, MA 02113]"
1,aGYdF_fNHDhFCAnXoTCkGA,boston-sail-loft-boston,Boston Sail Loft,https://s3-media0.fl.yelpcdn.com/bphoto/89XYbL...,False,https://www.yelp.com/biz/boston-sail-loft-bost...,2036,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.4,[pickup],$$,+16172277280,(617) 227-7280,2353.479804,42.362464,-71.050536,80 Atlantic Ave,,,Boston,02110,US,MA,"[80 Atlantic Ave, Boston, MA 02110]"
2,t_FFcwUutj9mIYKGw_gHsQ,the-salty-pig-boston,The Salty Pig,https://s3-media0.fl.yelpcdn.com/bphoto/y-_CvQ...,False,https://www.yelp.com/biz/the-salty-pig-boston?...,2075,"[{'alias': 'newamerican', 'title': 'New Americ...",4.2,[delivery],$$,+16175366200,(617) 536-6200,414.307221,42.346900,-71.076121,130 Dartmouth St,,,Boston,02116,US,MA,"[130 Dartmouth St, Boston, MA 02116]"
3,xh-i6tYmojnnUHrnkFnWdA,willow-and-ivy-boston,Willow & Ivy,https://s3-media0.fl.yelpcdn.com/bphoto/eIGgkA...,False,https://www.yelp.com/biz/willow-and-ivy-boston...,67,"[{'alias': 'newamerican', 'title': 'New Americ...",4.6,[],NaN,+16179334800,(617) 933-4800,713.579578,42.349126,-71.079754,65 Exeter St,NaN,,Boston,02116,US,MA,"[65 Exeter St, Boston, MA 02116]"
4,lsqYQYGfpe25wUZk9Wybwg,the-elephant-walk-boston-2,The Elephant Walk,https://s3-media0.fl.yelpcdn.com/bphoto/jnWAud...,False,https://www.yelp.com/biz/the-elephant-walk-bos...,368,"[{'alias': 'cambodian', 'title': 'Cambodian'},...",4.3,"[delivery, pickup]",$$,+16172471500,(617) 247-1500,744.978862,42.341160,-71.070578,1415 Washington St,,,Boston,02118,US,MA,"[1415 Washington St, Boston, MA 02118]"


In [10]:
df_restaurantes.to_csv("Restaurantes_Boston_API.csv")


##### This file was processed to match categories, weight scores, and apply additional transformations to align Yelp restaurant data with the client recommendation system.

**Processing steps:**
- Category normalization: Yelp categories mapped to dietary preference labels
- Price symbol conversion: $/$$/$$$/$$$$  →  numeric scale 1-4
- Score weighting: rating × 0.40 + review_count_normalized × 0.25 + price_match × 0.20 + category_match × 0.15
- Output: `yelp_treated_recommender_scale_1_4.csv` — clean dataset ready for recommendation engine
